In [2]:
import pandas as pd
import glob
import os

# Folder containing CSV files
folder = r"."

# Find all CSV files
files = glob.glob(os.path.join(folder, "*.csv"))

print(f"Found {len(files)} files")

# Load all files
dfs = []

for file in files:
    df = pd.read_csv(file)

    # Ensure correct column names
    expected = ["time", "open", "high", "low", "close", "Volume"]

    if list(df.columns) != expected:
        raise ValueError(
            f"{os.path.basename(file)} has unexpected columns:\n{df.columns}"
        )

    dfs.append(df)

# Combine
merged = pd.concat(dfs, ignore_index=True)

print(f"Rows before cleanup: {len(merged):,}")

# Sort by timestamp
merged = merged.sort_values("time")

# Remove duplicate timestamps
before = len(merged)

merged = merged.drop_duplicates(
    subset=["time"],
    keep="first"
)

after = len(merged)

print(f"Duplicate candles removed: {before - after:,}")

# Reset index
merged = merged.reset_index(drop=True)

# Check for missing timestamps
diffs = merged["time"].diff()

gaps = merged[diffs != 60]

if len(gaps) > 1:
    print("\nWARNING: Missing candle gaps detected\n")

    for idx in gaps.index[1:]:
        prev_time = merged.loc[idx - 1, "time"]
        curr_time = merged.loc[idx, "time"]

        missing = (curr_time - prev_time) // 60 - 1

        print(
            f"Gap between {prev_time} and {curr_time}"
            f" -> Missing {missing} minute candles"
        )
else:
    print("\nNo timestamp gaps detected")

# Save merged file
output_file = os.path.join(folder, "NQ_merged.csv")

merged.to_csv(output_file, index=False)

print(f"\nSaved: {output_file}")
print(f"Final rows: {len(merged):,}")

Found 18 files
Rows before cleanup: 352,969
Duplicate candles removed: 17


Gap between 1740175140 and 1740351600 -> Missing 2940 minute candles
Gap between 1740377700 and 1740377820 -> Missing 1 minute candles
Gap between 1740434340 and 1740438000 -> Missing 60 minute candles
Gap between 1740520740 and 1740524400 -> Missing 60 minute candles
Gap between 1740607140 and 1740610800 -> Missing 60 minute candles
Gap between 1740693540 and 1740697200 -> Missing 60 minute candles
Gap between 1740779940 and 1740956400 -> Missing 2940 minute candles
Gap between 1741039140 and 1741042800 -> Missing 60 minute candles
Gap between 1741125540 and 1741129200 -> Missing 60 minute candles
Gap between 1741211940 and 1741215600 -> Missing 60 minute candles
Gap between 1741298340 and 1741302000 -> Missing 60 minute candles
Gap between 1741384740 and 1741557600 -> Missing 2880 minute candles
Gap between 1741640340 and 1741644000 -> Missing 60 minute candles
Gap between 1741726740 and 1741730400 -> Missing

In [3]:
print("First timestamp:", merged["time"].min())
print("Last timestamp :", merged["time"].max())

First timestamp: 1740096000
Last timestamp : 1771624740


In [4]:
print(
    pd.to_datetime(
        merged["time"],
        unit="s",
        utc=True
    ).min()
)

print(
    pd.to_datetime(
        merged["time"],
        unit="s",
        utc=True
    ).max()
)

2025-02-21 00:00:00+00:00
2026-02-20 21:59:00+00:00
